In [0]:

# ============================================================
# CONSULTA 1: TOTAL DE CLUBES DE FÚTBOL
# Objetivo: contar los clubes registrados en la tabla Silver.
# ============================================================

query_1 = """
SELECT
    COUNT(*) AS total_football_clubs
FROM big_data_team3.fabrizio.clubes_futbol_silver
"""

# Ejecutar la consulta SQL mediante Spark
result_df_1 = spark.sql(query_1)

# Mostrar el resultado
display(result_df_1)


In [0]:

# ============================================================
# CONSULTA 2: EVENTOS INTERNACIONALES POR PAÍS
# Objetivo: identificar los países con más eventos registrados.
# ============================================================

query_2 = """
SELECT
    pais_evento AS country,
    COUNT(*) AS total_events
FROM big_data_team3.fabrizio.eventos_internacionales_silver
WHERE pais_evento IS NOT NULL
GROUP BY pais_evento
ORDER BY total_events DESC
"""

# Ejecutar la consulta mediante Spark
result_df_2 = spark.sql(query_2)

# Mostrar los resultados
display(result_df_2)


In [0]:

# ============================================================
# CONSULTA 3: EVENTOS INTERNACIONALES POR FEDERACIÓN
# Objetivo: identificar las federaciones con más eventos.
# ============================================================

query_3 = """
SELECT
    federacion AS federation,
    COUNT(*) AS total_events
FROM big_data_team3.fabrizio.eventos_internacionales_silver
WHERE federacion IS NOT NULL
GROUP BY federacion
ORDER BY total_events DESC
"""

# Ejecutar la consulta mediante Spark
result_df_3 = spark.sql(query_3)

# Mostrar los resultados
display(result_df_3)


In [0]:

# ============================================================
# CONSULTA 4: EVENTOS INTERNACIONALES POR AÑO
# Objetivo: analizar la cantidad de eventos por año.
# ============================================================

query_4 = """
SELECT
    YEAR(fecha_inicio) AS event_year,
    COUNT(*) AS total_events
FROM big_data_team3.fabrizio.eventos_internacionales_silver
WHERE fecha_inicio IS NOT NULL
GROUP BY YEAR(fecha_inicio)
ORDER BY event_year ASC
"""

# Ejecutar la consulta mediante Spark
result_df_4 = spark.sql(query_4)

# Mostrar los resultados
display(result_df_4)


In [0]:

# ============================================================
# CONSULTA 5: CONTEO DE REGISTROS Y ANÁLISIS DE NULOS
# Objetivo: evaluar el volumen y la calidad de los datos Silver.
# ============================================================

from pyspark.sql import functions as F

catalog = "big_data_team3"
schema = "fabrizio"

tables = [
    "clubes_futbol_silver",
    "clubes_renade_silver",
    "decan_silver",
    "eventos_internacionales_silver",
    "ligas_renade_silver",
    "pad_silver"
]

# ------------------------------------------------------------
# A. Contar los registros de cada tabla
# ------------------------------------------------------------

row_counts = []

for table in tables:
    full_table_name = f"{catalog}.{schema}.{table}"

    df = spark.table(full_table_name)
    total_rows = df.count()

    row_counts.append(
        (table, total_rows)
    )

row_counts_df = spark.createDataFrame(
    row_counts,
    ["table_name", "total_rows"]
)

print("TOTAL DE REGISTROS POR TABLA")
display(row_counts_df.orderBy(F.desc("total_rows")))

# ------------------------------------------------------------
# B. Analizar valores nulos de todas las columnas
# ------------------------------------------------------------

null_results = []

for table in tables:
    full_table_name = f"{catalog}.{schema}.{table}"
    df = spark.table(full_table_name)

    total_rows = df.count()

    for column_name in df.columns:
        null_count = (
            df.filter(F.col(column_name).isNull()).count()
        )

        null_percentage = (
            round(null_count * 100.0 / total_rows, 2)
            if total_rows > 0 else 0.0
        )

        null_results.append(
            (
                table,
                column_name,
                total_rows,
                null_count,
                null_percentage
            )
        )

null_results_df = spark.createDataFrame(
    null_results,
    [
        "table_name",
        "column_name",
        "total_rows",
        "null_count",
        "null_percentage"
    ]
)

print("ANÁLISIS DE VALORES NULOS")
display(
    null_results_df
    .filter(F.col("null_count") > 0)
    .orderBy(
        F.desc("null_percentage"),
        "table_name"
    )
)
